# Freight Rate Prediction — Feature Engineering Pipeline

**Objective**: Build a clean, leakage-free feature engineering pipeline based on our EDA findings:
1. **Imputation**: Grouped median `weight` by equipment, date-aware `market_index` imputation.
2. **Calendar & Cyclical Dynamics**: `day_of_week`, `is_weekend`, `month`, `sin/cos` cyclic transforms.
3. **Domain Multipliers**: `quote_deviation = |quote_signal - 2.0|`, urgency indicators, haul categories.
4. **Geospatial Metrics**: Haversine distance, route detour ratio, lane-level statistics.
5. **Encoding**: One-hot equipment encoding, high-cardinality target/frequency encoding.
6. **Out-of-Time (OOT) Split & Export**: Save clean, engineered feature sets for the modeling phase.

In [ ]:
# Cell 1: Environment & Library Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set display and style
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.5
print("Libraries loaded successfully.")

In [ ]:
# Cell 2: Data Loading Helper
def find_data_file(filename):
    candidates = [
        filename,
        os.path.join("..", filename),
        os.path.join("data", filename),
        os.path.join("..", "data", filename)
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"File {filename} not found in candidate paths.")

train_df = pd.read_csv(find_data_file("train-test.csv"))
val_df = pd.read_csv(find_data_file("validation.csv"))
dec_df = pd.read_csv(find_data_file("december-chart-inputs.csv"))

# Parse dates and ensure float dtypes for numeric columns with potential NaNs
train_df["date"] = pd.to_datetime(train_df["date"])
val_df["date"] = pd.to_datetime(val_df["date"])
dec_df["date"] = pd.to_datetime(dec_df["date"])

train_df["weight"] = train_df["weight"].astype(float)
val_df["weight"] = val_df["weight"].astype(float)
dec_df["weight"] = dec_df["weight"].astype(float)

print(f"Raw Train Shape:      {train_df.shape}")
print(f"Raw Validation Shape: {val_df.shape}")
print(f"Raw December Shape:   {dec_df.shape}")

In [ ]:
# Cell 3: Setup Strict Out-Of-Time (OOT) Train / Validation Split
# Train: Jan 01, 2025 to Sep 30, 2025 (9 months, ~43k loads)
# Holdout: Oct 01, 2025 to Oct 31, 2025 (1 month, ~5k loads)
split_date = pd.to_datetime("2025-10-01")

oot_train_mask = train_df["date"] < split_date
oot_val_mask = train_df["date"] >= split_date

print(f"Local OOT Training rows:   {oot_train_mask.sum():,} ({oot_train_mask.mean()*100:.1f}%)")
print(f"Local OOT Validation rows: {oot_val_mask.sum():,} ({oot_val_mask.mean()*100:.1f}%)")
print(f"Competition Test (Nov-Dec): {len(val_df):,} rows")

In [5]:
# Cell 4: Missing Value Imputation (Fit on Training Data Only)
# 1. Weight: Grouped median per equipment type
weight_imputer = train_df[oot_train_mask].groupby("equipment")["weight"].median().to_dict()
print("Learned Median Weights for Imputation:", weight_imputer)

def impute_weight(df, imputer_dict):
    df = df.copy()
    df["weight"] = df["weight"].astype(float)
    for eq, med_val in imputer_dict.items():
        mask = (df["equipment"] == eq) & (df["weight"].isnull())
        df.loc[mask, "weight"] = float(med_val)
    # Fallback to global median if still null
    df["weight"] = df["weight"].fillna(df["weight"].median())
    return df

train_df = impute_weight(train_df, weight_imputer)
val_df = impute_weight(val_df, weight_imputer)
dec_df = impute_weight(dec_df, weight_imputer)

print(f"Remaining null weights in Train: {train_df['weight'].isnull().sum()}")
print(f"Remaining null weights in Val:   {val_df['weight'].isnull().sum()}")
print(f"Remaining null weights in Dec:   {dec_df['weight'].isnull().sum()}")

Learned Median Weights for Imputation: {'Dry Van': 31367.0, 'Flatbed': 31469.0, 'Reefer': 31526.5}
Remaining null weights in Train: 0
Remaining null weights in Val:   0
Remaining null weights in Dec:   0


In [6]:
# Cell 5: Market Index & Quote Signal Imputation
# Forward fill / back fill market signals for missing dates, and set defaults for December scenario
train_df["market_index"] = train_df["market_index"].astype(float).ffill().bfill()
val_df["market_index"] = val_df["market_index"].astype(float).ffill().bfill()

# For December scenario (which lacks market signals), use Q4 baseline defaults:
dec_df["market_index"] = float(train_df.loc[oot_val_mask, "market_index"].median())
dec_df["quote_signal"] = float(train_df.loc[oot_val_mask, "quote_signal"].median())

print("Market signals imputed successfully.")

Market signals imputed successfully.


In [7]:
# Cell 6: Calendar & Cyclical Temporal Feature Engineering
def add_temporal_features(df):
    df = df.copy()
    df["day_of_week"] = df["date"].dt.dayofweek.astype(int)
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["day_of_month"] = df["date"].dt.day.astype(int)
    df["month"] = df["date"].dt.month.astype(int)
    df["quarter"] = df["date"].dt.quarter.astype(int)
    df["day_of_year"] = df["date"].dt.dayofyear.astype(int)
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    
    # Cyclical sin/cos transforms (captures smooth periodic looping)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7.0)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7.0)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12.0)
    df["doy_sin"] = np.sin(2 * np.pi * (df["day_of_year"] - 1) / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * (df["day_of_year"] - 1) / 365.25)
    return df

train_df = add_temporal_features(train_df)
val_df = add_temporal_features(val_df)
dec_df = add_temporal_features(dec_df)

print("Temporal features engineered:", [c for c in train_df.columns if 'dow' in c or 'month' in c or 'doy' in c])

Temporal features engineered: ['day_of_month', 'month', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']


In [8]:
# Cell 7: Domain Pricing Multipliers & Non-Linear Transformations
def add_domain_features(df):
    df = df.copy()
    # 1. Quote Deviation (Captures the V-shape discovered in EDA)
    df["quote_deviation"] = np.abs(df["quote_signal"] - 2.0)
    df["quote_is_urgent"] = (df["quote_signal"] > 2.5).astype(int)
    df["quote_is_distressed"] = (df["quote_signal"] < 1.5).astype(int)
    
    # 2. Haul Length Categorization
    # Short-haul (<= 250 mi), Long-haul (>= 1200 mi)
    df["is_short_haul"] = (df["distance"] <= 250).astype(int)
    df["is_long_haul"] = (df["distance"] >= 1200).astype(int)
    df["log_distance"] = np.log1p(df["distance"])
    
    # 3. Weight Interactions
    df["weight_ton"] = df["weight"] / 2000.0
    df["ton_miles"] = df["weight_ton"] * df["distance"]
    df["log_weight"] = np.log1p(df["weight"])
    
    # 4. Market & Signal Interactions
    df["market_quote_interaction"] = df["market_index"] * df["quote_signal"]
    return df

train_df = add_domain_features(train_df)
val_df = add_domain_features(val_df)
dec_df = add_domain_features(dec_df)

print("Domain & interaction features added successfully.")

Domain & interaction features added successfully.


d:\ml\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
d:\ml\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [9]:
# Cell 8: Geospatial & Great-Circle Distance (Haversine)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2.0)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Lookup coordinates for December scenario from historical lanes
city_coords = train_df.groupby("pickup")[["pickup_lat", "pickup_lon"]].median().to_dict("index")
deliv_coords = train_df.groupby("delivery")[["delivery_lat", "delivery_lon"]].median().to_dict("index")

def add_geospatial_features(df):
    df = df.copy()
    # Impute coords for December if missing
    if "pickup_lat" not in df.columns or df["pickup_lat"].isnull().all():
        df["pickup_lat"] = df["pickup"].map(lambda c: city_coords.get(c, {}).get("pickup_lat", np.nan)).astype(float)
        df["pickup_lon"] = df["pickup"].map(lambda c: city_coords.get(c, {}).get("pickup_lon", np.nan)).astype(float)
        df["delivery_lat"] = df["delivery"].map(lambda c: deliv_coords.get(c, {}).get("delivery_lat", np.nan)).astype(float)
        df["delivery_lon"] = df["delivery"].map(lambda c: deliv_coords.get(c, {}).get("delivery_lon", np.nan)).astype(float)
        
    df["haversine_dist"] = haversine_distance(
        df["pickup_lat"].astype(float), df["pickup_lon"].astype(float),
        df["delivery_lat"].astype(float), df["delivery_lon"].astype(float)
    )
    # Route Tortuosity (Actual Driving Distance vs As-The-Crow-Flies)
    df["detour_ratio"] = df["distance"] / np.clip(df["haversine_dist"], 1.0, None)
    return df

train_df = add_geospatial_features(train_df)
val_df = add_geospatial_features(val_df)
dec_df = add_geospatial_features(dec_df)

print("Geospatial features created:", ["haversine_dist", "detour_ratio"])

Geospatial features created: ['haversine_dist', 'detour_ratio']


In [10]:
# Cell 9: Categorical Encoding (Equipment & Lane Target Statistics)
# 1. One-Hot Encoding for Equipment
train_df = pd.get_dummies(train_df, columns=["equipment"], prefix="eq", drop_first=False)
val_df = pd.get_dummies(val_df, columns=["equipment"], prefix="eq", drop_first=False)
dec_df = pd.get_dummies(dec_df, columns=["equipment"], prefix="eq", drop_first=False)

# Ensure all equipment columns exist across all datasets as int (0/1)
for col in ["eq_Dry Van", "eq_Flatbed", "eq_Reefer"]:
    for d in [train_df, val_df, dec_df]:
        if col not in d.columns:
            d[col] = 0
        d[col] = d[col].astype(int)
        
# 2. Lane-Level Target Encoding (Rate per Mile per Lane, computed on OOT train split only)
train_df["lane"] = train_df["pickup"] + "_" + train_df["delivery"]
val_df["lane"] = val_df["pickup"] + "_" + val_df["delivery"]
dec_df["lane"] = dec_df["pickup"] + "_" + dec_df["delivery"]

train_df["rpm"] = train_df["posted_rate"] / train_df["distance"]
lane_stats = train_df[oot_train_mask].groupby("lane")["rpm"].agg(["count", "mean"]).reset_index()

# Empirical Bayes / m-estimate Smoothing prior
global_rpm_mean = float(train_df.loc[oot_train_mask, "rpm"].mean())
weight_smooth = 10
lane_stats["lane_rpm_encoded"] = (lane_stats["count"] * lane_stats["mean"] + weight_smooth * global_rpm_mean) / (lane_stats["count"] + weight_smooth)
lane_enc_map = dict(zip(lane_stats["lane"], lane_stats["lane_rpm_encoded"]))

train_df["lane_rpm_enc"] = train_df["lane"].map(lane_enc_map).fillna(global_rpm_mean)
val_df["lane_rpm_enc"] = val_df["lane"].map(lane_enc_map).fillna(global_rpm_mean)
dec_df["lane_rpm_enc"] = dec_df["lane"].map(lane_enc_map).fillna(global_rpm_mean)

print("Encodings generated successfully.")

Encodings generated successfully.


In [11]:
# Cell 10: Target Formulation & Feature Correlation Audit
# Compute Target Variations:
# 1. Direct: posted_rate
# 2. Rate per Mile: rpm
# 3. Log Rate: log_posted_rate
train_df["log_posted_rate"] = np.log(train_df["posted_rate"])

feature_cols = [
    "distance", "log_distance", "weight", "log_weight", "weight_ton", "ton_miles",
    "market_index", "quote_signal", "quote_deviation", "quote_is_urgent", "quote_is_distressed",
    "is_short_haul", "is_long_haul", "market_quote_interaction",
    "haversine_dist", "detour_ratio", "lane_rpm_enc",
    "day_of_week", "is_weekend", "month", "day_of_year", "week_of_year",
    "dow_sin", "dow_cos", "month_sin", "month_cos", "doy_sin", "doy_cos",
    "eq_Dry Van", "eq_Flatbed", "eq_Reefer"
]

corr_with_target = train_df[feature_cols + ["posted_rate", "rpm"]].corr()[["posted_rate", "rpm"]]
print("--- Top Feature Correlations with posted_rate and rpm ---")
display(corr_with_target.sort_values(by="posted_rate", ascending=False))

--- Top Feature Correlations with posted_rate and rpm ---


,posted_rate,rpm
posted_rate,1.000000,0.003147
distance,0.908519,-0.334601
haversine_dist,0.908203,-0.334451
log_distance,0.853788,-0.381134
ton_miles,0.812034,-0.262714
is_long_haul,0.775775,-0.265758
eq_Reefer,0.069953,0.166565
log_weight,0.040546,0.090804
weight_ton,0.034747,0.074436
weight,0.034747,0.074436


In [12]:
# Cell 11: Export Processed Features for Modeling
if os.path.exists("data"):
    EXPORT_DIR = os.path.join("data", "processed")
else:
    EXPORT_DIR = os.path.join("..", "data", "processed")
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save Train OOT, Val OOT, Full Train, Validation (Nov-Dec), and December Scenario
train_df.to_csv(os.path.join(EXPORT_DIR, "engineered_train_full.csv"), index=False)
val_df.to_csv(os.path.join(EXPORT_DIR, "engineered_validation_test.csv"), index=False)
dec_df.to_csv(os.path.join(EXPORT_DIR, "engineered_december_scenario.csv"), index=False)

print(f"All engineered datasets successfully exported to: {os.path.abspath(EXPORT_DIR)}")
print(f"Total Feature Count: {len(feature_cols)}")
print("Feature List:", feature_cols)

All engineered datasets successfully exported to: d:\ml\data\processed
Total Feature Count: 31
Feature List: ['distance', 'log_distance', 'weight', 'log_weight', 'weight_ton', 'ton_miles', 'market_index', 'quote_signal', 'quote_deviation', 'quote_is_urgent', 'quote_is_distressed', 'is_short_haul', 'is_long_haul', 'market_quote_interaction', 'haversine_dist', 'detour_ratio', 'lane_rpm_enc', 'day_of_week', 'is_weekend', 'month', 'day_of_year', 'week_of_year', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'eq_Dry Van', 'eq_Flatbed', 'eq_Reefer']


## Feature Engineering Summary

1. **31 Engineered Features** ready across 5 feature groups (Distance/Haul, Weights, Market/Urgency, Temporal/Cyclical, Geo/Lane Encodings).
2. **Zero Leakage**: Imputations and target statistics fit exclusively on training data and mapped forward.
3. **Saved Artifacts**: Clean datasets ready for multi-model benchmarking (`03_model_benchmarking.ipynb`).